# SubaRAG data pilot
Run this notebook in a Google Colab CPU runtime. It prepares a bounded English/Hindi/Marathi sample; no embeddings or RAG are run.

Upload the committed source archive `SubaRAG-source.zip` when prompted. The archive must contain `SubaRAG/backend/pyproject.toml`. Do not upload `.env` files or keys. Reading the public source needs no Hugging Face token. See `docs/data.md` for the artifact contract and the distinction between this pilot and a full build.

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, io, subprocess, sys, tempfile, zipfile

def run_logged(arguments):
    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code != 0:
        raise RuntimeError(f'Command stopped with exit code {code}. Share the progress lines above; do not delete partial output.')

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one SubaRAG source ZIP.')
archive_bytes = next(iter(uploaded.values()))
source_root = Path(tempfile.mkdtemp(prefix='subarag-source-', dir='/content'))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    reader_source = archive.read('SubaRAG/backend/src/subarag/data/source.py').decode('utf-8')
    if 'parquet-iter-batches-v1' not in reader_source:
        raise ValueError('This ZIP contains the old reader. Upload the corrected source ZIP.')
    for member in archive.infolist():
        destination = (source_root / member.filename).resolve()
        if not destination.is_relative_to(source_root) or (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Archive has an unsafe path or symlink.')
    archive.extractall(source_root)
backend = source_root / 'SubaRAG' / 'backend'
if not (backend / 'pyproject.toml').is_file():
    raise ValueError('Expected SubaRAG/backend/pyproject.toml in source archive.')
run_logged([sys.executable, '-m', 'pip', 'install', f'{backend}[data]'])
run_logged([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', str(backend)])
reader_hash = hashlib.sha256((backend / 'src/subarag/data/source.py').read_bytes()).hexdigest()
verify_reader = "import hashlib, sys; from pathlib import Path; from subarag.data import source; print('Loaded reader:', source.__file__, flush=True); assert hashlib.sha256(Path(source.__file__).read_bytes()).hexdigest() == sys.argv[1], 'Installed reader differs from uploaded source'; print('Reader verified:', source.READER_VERSION, flush=True)"
run_logged([sys.executable, '-u', '-c', verify_reader, reader_hash])


In [ ]:
OUTPUT = Path(tempfile.mkdtemp(prefix='subarag-data-pilot-', dir='/content'))
print(f'Output folder: {OUTPUT}', flush=True)
REVISION = 'bf5cdc1f26e581e519018e434db14edd1b77602b'
run_logged([sys.executable, '-u', '-m', 'subarag.data', 'prepare',
                '--output', str(OUTPUT), '--revision', REVISION,
                '--languages', 'en', 'hi', 'mr', '--rows-per-source', '1000'])
run_logged([sys.executable, '-u', '-m', 'subarag.data', 'verify',
                str(OUTPUT / 'manifest.json')])


## Review the result
Check source, document, evaluation, and rejection counts in the output above. Download the small manifest below and share it with the project. Keep the corpus files in this runtime for the subsequent authorized upload to a private artifact repository. Colab disk is temporary; a disconnected runtime may lose these files. Uploading/publishing artifacts and starting a full job require a separate step.

The completed directory includes JSONL shards, `manifest.json`, and a local `_work` SQLite store. Only the manifest and its listed shards belong in the eventual artifact repository. No account login or token is requested by this notebook beyond your Colab session.

In [ ]:
files.download(str(OUTPUT / 'manifest.json'))
